In [1]:
import pandas as pd
import sqlite3

df = pd.read_csv('OneDrive/Desktop/FMCG_Project/outputs/retail_clean.csv')
df['invoicedate'] = pd.to_datetime(df['invoicedate'])

# Create SQLite database
conn = sqlite3.connect('OneDrive/Desktop/FMCG_Project/outputs/fmcg.db')

# Write dataframe to SQL table
df.to_sql('transactions', conn, if_exists='replace', index=False)

# Check
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)

print("✓ Database created")
print(f"✓ Rows loaded: {pd.read_sql('SELECT COUNT(*) as cnt FROM transactions', conn).iloc[0,0]:,}")
print(f"✓ Tables: {tables['name'].tolist()}")


✓ Database created
✓ Rows loaded: 805,549
✓ Tables: ['rfm', 'transactions']


In [2]:
# Always explore before querying
q = """
SELECT 
    COUNT(*)                        as total_rows,
    COUNT(DISTINCT customer_id)     as unique_customers,
    COUNT(DISTINCT invoice)         as unique_invoices,
    COUNT(DISTINCT stockcode)       as unique_products,
    ROUND(SUM(revenue), 2)          as total_revenue
FROM transactions
"""
print(pd.read_sql(q, conn).to_string(index=False))

 total_rows  unique_customers  unique_invoices  unique_products  total_revenue
     805549              5878            36969             4631    17743429.18


In [3]:
# Query 1 revenue by country
q = """
SELECT 
    country,
    COUNT(DISTINCT customer_id)     as customers,
    COUNT(DISTINCT invoice)         as orders,
    ROUND(SUM(revenue), 2)          as total_revenue,
    ROUND(SUM(revenue) * 100.0 / 
        (SELECT SUM(revenue) FROM transactions), 1) as revenue_pct
FROM transactions
GROUP BY country
ORDER BY total_revenue DESC
LIMIT 10
"""
result = pd.read_sql(q, conn)
print(result.to_string(index=False))
result.to_csv('OneDrive/Desktop/FMCG_Project/outputs/sql_revenue_by_country.csv', index=False)
print("\n✓ Saved sql_revenue_by_country.csv")

       country  customers  orders  total_revenue  revenue_pct
United Kingdom       5350   33541    14723147.52         83.0
          EIRE          5     567      621631.11          3.5
   Netherlands         22     228      554232.34          3.1
       Germany        107     789      431262.46          2.4
        France         95     614      355257.47          2.0
     Australia         15      95      169968.11          1.0
         Spain         41     154      109178.53          0.6
   Switzerland         22      90      100365.34          0.6
        Sweden         19     104       91549.72          0.5
       Denmark         12      43       69862.19          0.4

✓ Saved sql_revenue_by_country.csv


In [4]:
# Query 2 Monthly revenue trends
q = """
SELECT 
    year_month,
    COUNT(DISTINCT customer_id)     as active_customers,
    COUNT(DISTINCT invoice)         as total_orders,
    ROUND(SUM(revenue), 2)          as monthly_revenue,
    ROUND(AVG(revenue), 2)          as avg_order_value
FROM transactions
GROUP BY year_month
ORDER BY year_month
"""
result = pd.read_sql(q, conn)
print(result.to_string(index=False))
result.to_csv('OneDrive/Desktop/FMCG_Project/outputs/sql_monthly_trend.csv', index=False)
print("\n✓ Saved sql_monthly_trend.csv")

year_month  active_customers  total_orders  monthly_revenue  avg_order_value
   2009-12               955          1512        686654.16            22.33
   2010-01               720          1011        557319.06            25.59
   2010-02               772          1104        506371.07            21.67
   2010-03              1057          1524        699608.99            21.66
   2010-04               942          1329        594609.19            21.83
   2010-05               966          1377        599985.79            20.95
   2010-06              1041          1497        639066.58            20.49
   2010-07               928          1381        591636.74            21.89
   2010-08               911          1293        604242.65            22.89
   2010-09              1145          1689        831615.00            24.04
   2010-10              1497          2133       1036680.00            20.92
   2010-11              1607          2587       1172336.04            19.44

In [5]:
# Query 3: Top 10 products by revenue
q = """
SELECT 
    stockcode,
    description,
    COUNT(DISTINCT invoice)         as times_ordered,
    SUM(quantity)                   as total_qty_sold,
    ROUND(SUM(revenue), 2)          as total_revenue,
    ROUND(AVG(price), 2)            as avg_unit_price
FROM transactions
GROUP BY stockcode, description
ORDER BY total_revenue DESC
LIMIT 10
"""
result = pd.read_sql(q, conn)
print(result.to_string(index=False))
result.to_csv('OneDrive/Desktop/FMCG_Project/outputs/sql_top_products.csv', index=False)
print("\n✓ Saved sql_top_products.csv")

stockcode                        description  times_ordered  total_qty_sold  total_revenue  avg_unit_price
    22423           REGENCY CAKESTAND 3 TIER           3317           24899      286486.30           12.46
   85123A WHITE HANGING HEART T-LIGHT HOLDER           4888           93640      252072.46            2.87
    23843        PAPER CRAFT , LITTLE BIRDIE              1           80995      168469.60            2.08
        M                             Manual            620            9803      152340.57          206.44
   85099B            JUMBO BAG RED RETROSPOT           2612           75759      136980.08            1.97
    84879      ASSORTED COLOUR BIRD ORNAMENT           2652           79913      127074.17            1.68
     POST                            POSTAGE           1803            5333      126563.04           29.75
    47566                      PARTY BUNTING           2077           23607      103880.23            4.77
    23166     MEDIUM CERAMIC TOP STOR

In [6]:
# Query 4: Top 10 customers by revenue (with RFM segment)
# Load RFM scores to join
rfm = pd.read_csv('OneDrive/Desktop/FMCG_Project/outputs/rfm_scores.csv')
rfm.to_sql('rfm', conn, if_exists='replace', index=False)

q = """
SELECT 
    t.customer_id,
    t.country,
    COUNT(DISTINCT t.invoice)       as total_orders,
    ROUND(SUM(t.revenue), 2)        as total_spend,
    r.segment
FROM transactions t
LEFT JOIN rfm r ON t.customer_id = r.customer_id
GROUP BY t.customer_id, t.country, r.segment
ORDER BY total_spend DESC
LIMIT 10
"""
result = pd.read_sql(q, conn)
print(result.to_string(index=False))
result.to_csv('OneDrive/Desktop/FMCG_Project/outputs/sql_top_customers.csv', index=False)
print("\n✓ Saved sql_top_customers.csv")

 customer_id        country  total_orders  total_spend      segment
       18102 United Kingdom           145    608821.65     Champion
       14646    Netherlands           151    528602.52     Champion
       14156           EIRE           156    313946.37     Champion
       14911           EIRE           398    295972.63     Champion
       17450 United Kingdom            51    246973.09     Champion
       13694 United Kingdom           143    196482.81     Champion
       17511 United Kingdom            60    175603.55     Champion
       16446 United Kingdom             2    168472.50 New Customer
       16684 United Kingdom            55    147142.77     Champion
       12415      Australia            28    144458.37     Champion

✓ Saved sql_top_customers.csv


In [7]:
#Basket Analysis
q = """
SELECT 
    a.description        as product_a,
    b.description        as product_b,
    COUNT(*)             as times_bought_together
FROM transactions a
JOIN transactions b 
    ON  a.invoice    = b.invoice          -- same order
    AND a.stockcode  < b.stockcode        -- avoid duplicates & self-pairs
WHERE a.description IS NOT NULL
  AND b.description IS NOT NULL
  AND a.description != b.description
GROUP BY a.description, b.description
HAVING COUNT(*) >= 20                     -- only pairs that appear 20+ times
ORDER BY times_bought_together DESC
LIMIT 20
"""
result = pd.read_sql(q, conn)
print(result.to_string(index=False))
result.to_csv('OneDrive/Desktop/FMCG_Project/outputs/sql_product_pairs.csv', index=False)
print("\n✓ Saved sql_product_pairs.csv")

                         product_a                          product_b  times_bought_together
  RED HANGING HEART T-LIGHT HOLDER WHITE HANGING HEART T-LIGHT HOLDER                   1343
 WOODEN PICTURE FRAME WHITE FINISH        WOODEN FRAME ANTIQUE WHITE                    1138
             HEART OF WICKER SMALL              HEART OF WICKER LARGE                   1037
    SWEETHEART CERAMIC TRINKET BOX     STRAWBERRY CERAMIC TRINKET BOX                    979
          HOME BUILDING BLOCK WORD           LOVE BUILDING BLOCK WORD                    945
        ALARM CLOCK BAKELIKE GREEN          ALARM CLOCK BAKELIKE RED                     896
   GREEN REGENCY TEACUP AND SAUCER     PINK REGENCY TEACUP AND SAUCER                    894
           LUNCH BAG  BLACK SKULL.         LUNCH BAG SPACEBOY DESIGN                     887
   PAPER CHAIN KIT 50'S CHRISTMAS   PAPER CHAIN KIT VINTAGE CHRISTMAS                    887
   GREEN REGENCY TEACUP AND SAUCER   ROSES REGENCY TEACUP AND SAUCER  

In [8]:
#Closing Connection
conn.close()
print("✓ Database connection closed")
print("\n=== FILES SAVED ===")
print("sql_revenue_by_country.csv")
print("sql_monthly_trend.csv")
print("sql_top_products.csv")
print("sql_top_customers.csv")
print("sql_product_pairs.csv")
print("fmcg.db")

✓ Database connection closed

=== FILES SAVED ===
sql_revenue_by_country.csv
sql_monthly_trend.csv
sql_top_products.csv
sql_top_customers.csv
sql_product_pairs.csv
fmcg.db
